In [219]:
# pip install python-dotenv
# pip install dremio-simple-query
# pip install statsmodels
# pip install joblib
# pip install lightgbm
# pip install scikit-learn 
# pip install seaborn


In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
from dataclasses import dataclass
import joblib
from lightgbm import LGBMRegressor ,early_stopping
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
from dremio_simple_query.connectv2 import DremioConnection
import argparse
import os
import seaborn as sns
import matplotlib.pyplot as plt
dremio = DremioConnection(
    location="grpc://dremio:32010",  
    username="mofah",
    password="fahmy12345",
)

In [13]:
features = [
    # Categories 
    'category', 'sub_category', 'store_region',
    
    # Prices & Margins
    'log_unit_retail', 'avg_unit_cost', 'avg_competitor_price',
    'min_competitor_price', 'max_competitor_price',
    'competitor_price_ratio', 'price_difference', 'competitor_price_gap_pct',
    'current_profit_margin_pct', 'is_higher_than_competitor', 'is_undercut',
    
    # Lags & Rollings
    'quantity_lag_1_week', 'quantity_lag_4_weeks', 
    'quantity_rolling_avg_4_weeks', 'price_change_pct_weekly', 'price_rolling_avg_4_weeks',
    
    # Seasonality
    'month', 'quarter', 'week_of_year'
]

categorical_cols = ['category', 'sub_category', 'store_region', 'is_higher_than_competitor', 'is_undercut']
    
target =  "log_quantity"   

# 1- train_demand_model

In [32]:
@dataclass
class TrainResult:
    model: LGBMRegressor
    test_mae: float
    train_mae: float
    naive_elasticity: float
    # simple log-log OLS coefficient, sanity-check only
    # LightGBM maps pandas categorical columns to integer codes internally
    # based on the exact set of categories seen at fit time. If prediction-
    # time data has a different (even just differently-ordered) set of
    # categories, the same code can silently refer to a different category -
    # wrong predictions with no error raised. Persisting these dtypes and
    # re-applying them at prediction time (see price_optimizer.py) is what
    # prevents that.
    category_dtypes: dict
        
def read_data() -> pd.DataFrame:
    df = dremio.toPandas("SELECT * FROM nessie.marts.ml_dynamic_competitive_pricing")
    return df
    
def Feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['product_id', 'store_key', 'sales_week']).reset_index(drop=True)
    grouped = df.groupby(['product_id', 'store_key'])

    #df['log_quantity'] = np.log1p(df['quantity_sold'])
    df['log_unit_retail'] = np.log(df['avg_unit_retail'])
    #df['total_revenue'] = df['quantity_sold'] * df['avg_unit_retail']
    #df['units_per_transaction'] = df['quantity_sold'] / np.maximum(df['total_transactions'], 1)
    df['competitor_price_ratio'] = df['avg_unit_retail'] / df['avg_competitor_price']
    df['price_difference'] = df['avg_unit_retail'] - df['avg_competitor_price']
    df['competitor_price_gap_pct'] = df['price_difference'] / df['avg_competitor_price']

    # No fillna(0) - a missing margin is unknown, not zero. LightGBM
    # handles real NaN natively; manufacturing 0 just teaches the model
    # a false "no margin" signal for rows where margin is simply unknown.
    df['current_profit_margin_pct'] = (df['avg_unit_retail'] - df['avg_unit_cost']) / df['avg_unit_retail']
    #df['current_profit_margin_pct'] = df['quantity_sold'] * (df['avg_unit_retail'] - df['avg_unit_cost'])
    

    # np.nan, not pd.NA - keeps this a clean float64 column instead of
    # collapsing to object dtype, which is what caused the crash.
    df['is_higher_than_competitor'] = np.where(
        df['avg_competitor_price'].isna(), np.nan,
        (df['avg_unit_retail'] > df['avg_competitor_price']).astype(float)
    )
    df['is_undercut'] = np.where(
        df['min_competitor_price'].isna(), np.nan,
        (df['avg_unit_retail'] > df['min_competitor_price']).astype(float)
    )

    # No fillna(0) on lags - a real gap in history, left as NaN, is what
    # LightGBM is designed to split around correctly.
    df['quantity_lag_1_week'] = grouped['quantity_sold'].shift(1)
    df['quantity_lag_4_weeks'] = grouped['quantity_sold'].shift(4)
    df['price_change_pct_weekly'] = grouped['avg_unit_retail'].pct_change(periods=1)

    df['quantity_rolling_avg_4_weeks'] = (
        df.groupby(['product_id', 'store_key'])['quantity_sold']
          .transform(lambda s: s.shift(1).rolling(window=4, min_periods=1).mean())
    )
    # No fillna(df['avg_unit_retail']) - that fallback was silently
    # setting "4-week rolling avg price" equal to "today's price" for new
    # product-stores, artificially zeroing out any price-change signal
    # exactly where the model most needs to see "no history yet" honestly.
    df['price_rolling_avg_4_weeks'] = (
        df.groupby(['product_id', 'store_key'])['avg_unit_retail']
          .transform(lambda s: s.shift(1).rolling(window=4, min_periods=1).mean())
    )

    df['month'] = df['sales_week'].dt.month
    df['quarter'] = df['sales_week'].dt.quarter
    df['week_of_year'] = df['sales_week'].dt.isocalendar().week.astype(int)

    return df

def prepare_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    float_cols = [
        'avg_unit_retail',
        'avg_unit_cost',
        'avg_competitor_price',
        'min_competitor_price',
        'max_competitor_price' ]
    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

    
    df['sales_week'] = pd.to_datetime(df['sales_week'])

    return df
def _prepare_after_feature_eng(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in categorical_cols:
        df[col] = df[col].astype('category')
    df['log_quantity'] = np.log1p(df['quantity_sold'])    
    df = df.dropna(subset=['quantity_lag_4_weeks']).reset_index(drop=True)
    return df

def _naive_elasticity_check(df: pd.DataFrame) -> float:
    """
    Fits a plain log-log OLS (quantity ~ price only) as an interpretable
    sanity-check baseline. If this coefficient and the ML model's implied
    elasticity (see evaluate_elasticity in price_optimizer.py) disagree by
    a wide margin, that's a signal to investigate before trusting the ML
    model's price recommendations - not something to silently ignore.
    """
    X = sm.add_constant(df["log_unit_retail"])
    y = df["log_quantity"]
    result = sm.OLS(y, X).fit()
    return float(result.params["log_unit_retail"])


def train(df: pd.DataFrame, test_days: int = 28) -> TrainResult:
    df = _prepare_after_feature_eng(df)
    df = df.sort_values("sales_week")

    cutoff = df["sales_week"].max() - pd.Timedelta(days=test_days)
    train_df = df[df["sales_week"] <= cutoff]
    test_df = df[df["sales_week"] > cutoff]
    x_train = train_df[features]
    y_train = train_df[target]
    x_test = test_df[features]
    y_test = test_df[target]

    if train_df.empty or test_df.empty:
        raise ValueError(
            f"Time-based split produced an empty set (test_days={test_days}). "
            "Check that pricing_features actually spans more than test_days "
            "of history before training.")

    monotonic = [-1 if col == "log_unit_retail" else 0 for col in features ]

    model = LGBMRegressor(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            monotone_constraints=monotonic,
            monotone_constraints_method="advanced",
            random_state=42)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        model.fit( x_train , y_train,categorical_feature=categorical_cols)

    train_preds = model.predict(x_train)
    test_preds = model.predict(x_test)
    
    train_mae = mean_absolute_error(y_train,train_preds)
    test_mae = mean_absolute_error(y_test, test_preds)
    #test_rmse = float(np.sqrt(mean_squared_error(y_test, test_preds)))
    #r2 = r2_score(y_test, preds)

    naive_elasticity = _naive_elasticity_check(train_df)

    category_dtypes = {col: train_df[col].dtype for col in categorical_cols}

    return TrainResult(
        model=model,
        train_mae=train_mae,
        test_mae=test_mae,
        #r2 = r2,
        naive_elasticity=naive_elasticity,
        category_dtypes=category_dtypes,
    )

def save(result: TrainResult, path: str = "demand_model.joblib") -> None:

    joblib.dump({"model": result.model, "category_dtypes": result.category_dtypes}, path)


def load(path: str) -> tuple:
    """Returns (model, category_dtypes). Use this instead of joblib.load directly."""
    bundle = joblib.load(path)
    return bundle["model"], bundle["category_dtypes"]



# 2- price_optimizer

In [15]:
@dataclass
class PriceRecommendation:
    product_id: int
    store_id: int
    current_price: float
    current_weekly_quantity: int
    current_weekly_margin: float
    recommended_price: float
    predicted_weekly_quantity: float
    predicted_weekly_margin: float
    candidates: pd.DataFrame
        

def fix_prediction_dtypes(rows: pd.DataFrame, category_dtypes: dict) -> pd.DataFrame:
    """
    base_row.to_frame().T turns a single pandas Series into a one-row
    DataFrame - and since a Series holds one dtype for its whole length,
    EVERY column comes out as generic `object`, not just the categoricals.
    LightGBM rejects object-dtyped numeric columns outright (as our own
    test caught). Two separate fixes needed:
    1. Categorical columns get the exact dtype (same categories, same
       order) captured at training time - re-inferring categories fresh
       from a single repeated row would only ever see one category level,
       which LightGBM either rejects or, worse, could silently map to the
       wrong code if it doesn't reject it.
    2. Every other feature column gets coerced back to numeric explicitly -
       object dtype survives even for what were originally float/int/bool
       values, since the Series-transpose step erases that distinction.
    """
    rows = rows.copy()
    for col, dtype in category_dtypes.items():
        rows[col] = rows[col].astype(dtype)

    numeric_columns = [c for c in features if c not in category_dtypes]
    for col in numeric_columns:
        rows[col] = pd.to_numeric(rows[col], errors="raise")

    return rows    

def evaluate_elasticity( model, base_row: pd.Series, price_range: tuple[float, float], category_dtypes: dict, n_points: int = 20 ) -> float:
    """
    Reads the model's own implied elasticity at a specific product-store's
    current context, by finite-differencing predicted demand across a
    small price range. Compare this against train_demand_model.py's naive
    OLS baseline before trusting a recommendation - large disagreement
    means the model is extrapolating unreliably for this product-store,
    not that it has found a genuinely different elasticity.
    """
    prices = np.linspace(price_range[0], price_range[1], n_points)
    rows = pd.concat([base_row.to_frame().T] * n_points, ignore_index=True)
    rows["log_unit_retail"] = np.log(prices)
    rows = _fix_prediction_dtypes(rows, category_dtypes)
    preds = model.predict(rows[features])
    # elasticity = d(log quantity) / d(log price)
    slope = np.polyfit(np.log(prices), preds, 1)[0]
    return float(slope)


def recommend_price(model,base_row: pd.Series,category_dtypes: dict,min_margin_pct: float = 0.10,max_competitive_gap_pct: float = 0.05,n_candidates: int = 50) -> PriceRecommendation:
    """
    base_row: one row of engineered features for a specific
    (product_key, store_key), at its most recent known state - i.e. the
    latest row from pricing_features for that pair. All non-price features
    are held fixed at their current values; only price is varied across
    the candidate grid.

    Constraints, both enforced as hard filters (candidates violating either
    are excluded outright, not merely penalized):
    - min_margin_pct: price must clear unit_cost by at least this margin.
      This is a floor, not a target - it exists to prevent the optimizer
      from ever recommending a price that loses money or breaks a
      contractual minimum margin, regardless of what the demand curve says
      would maximize predicted quantity.
    - max_competitive_gap_pct: price must stay within this fraction of the
      last known competitor price. This is a business guardrail, not
      something derived from the demand model - it exists because pure
      margin-maximization without a competitive constraint can recommend
      prices far above market that the demand model has never actually
      observed and is extrapolating into blindly.
    """
    unit_cost = base_row["avg_unit_cost"]
    competitor_price = base_row["avg_competitor_price"]
    current_price = base_row["avg_unit_retail"]

    price_floor = unit_cost * (1 + min_margin_pct)
    if pd.notna(competitor_price):
        competitive_low = competitor_price * (1 - max_competitive_gap_pct)
        competitive_high = competitor_price * (1 + max_competitive_gap_pct)
    else:
        # No competitor price known for this product - fall back to a
        # wider band around current price rather than silently ignoring
        # the constraint entirely. Flag this explicitly in the output
        # rather than pretending the constraint was meaningfully applied.
        competitive_low = current_price * 0.85
        competitive_high = current_price * 1.15

    grid_low = max(price_floor, competitive_low)
    grid_high = max(grid_low * 1.01, competitive_high)  # guard against inverted/degenerate range

    candidate_prices = np.linspace(grid_low, grid_high, n_candidates)

    rows = pd.concat([base_row.to_frame().T] * n_candidates, ignore_index=True)
    rows["log_unit_retail"] = np.log(candidate_prices)
    rows["avg_unit_retail"] = candidate_prices
    rows['competitor_price_ratio'] = candidate_prices / rows['avg_competitor_price']
    rows['price_difference'] = candidate_prices - rows['avg_competitor_price']
    rows['competitor_price_gap_pct'] = rows['price_difference'] / rows['avg_competitor_price']
    rows['current_profit_margin_pct'] = (candidate_prices - rows['avg_unit_cost']) / candidate_prices
    
   
    rows['is_higher_than_competitor'] = np.where(
        rows['avg_competitor_price'].isna(), np.nan,
        ( candidate_prices > rows['avg_competitor_price']).astype(float))
    rows['is_undercut'] = np.where(
        rows['min_competitor_price'].isna(), np.nan,
        (candidate_prices > rows['min_competitor_price']).astype(float))    
    
    rows = fix_prediction_dtypes(rows, category_dtypes)

    predicted_log_quantity = model.predict(rows[features])
    predicted_quantity = np.expm1(predicted_log_quantity).clip(min=0)
    predicted_margin = predicted_quantity * (candidate_prices - unit_cost)
    

    candidates_df = pd.DataFrame({
        "price": candidate_prices,
        "predicted_quantity": predicted_quantity,
        "predicted_margin": predicted_margin})

    best_idx = candidates_df["predicted_margin"].idxmax()
    best = candidates_df.loc[best_idx]
 
    return PriceRecommendation(
        product_id=base_row["product_id"],
        store_id=base_row["store_id"],
        current_price=current_price,
        current_weekly_quantity=base_row["quantity_sold"],
        current_weekly_margin= base_row["current_profit_margin_pct"],
        recommended_price=float(best["price"]),
        predicted_weekly_quantity=float(best["predicted_quantity"]),
        predicted_weekly_margin=float(best["predicted_margin"]),
        candidates=candidates_df)


# run_pricing_pipeline

In [25]:
def train_and_save(model_path: str) -> None:
    df = read_data() 
    df_pre = prepare_raw_data(df)
    df_features = Feature_engineering(df_pre)
    result = train(df_features)

    print(f"Train MAE (log-quantity space): {result.train_mae:.4f}")
    print(f"Test MAE (log-quantity space): {result.test_mae:.4f}")
    #print(f"R² Score: {result.r2 * 100:.2f}%")
    print(f"Naive log-log OLS elasticity (sanity check): {result.naive_elasticity:.4f}")
    
    #save(result ,path=model_path)

def score_all(model_path: str, output_path: str, min_margin_pct: float, max_competitive_gap_pct: float) -> None:
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"No trained model found at {model_path}. Run train_and_save "
            "first - scoring should never silently fall back to training "
            "inline, since that would retrain on a schedule meant only for "
            "scoring, hiding when the model actually last changed."
        )
    model, category_dtypes = load(model_path)
    df = read_data() 
    df_pre = prepare_raw_data(df)
    features_df = Feature_engineering(df_pre)
    

    # Score using each product-store's most recent row only - recommending
    # a price is inherently a "given today's context" decision, not
    # something to backfill across history.
    latest = (
        features_df.sort_values("sales_week")
        .groupby(["product_id", "store_id"], as_index=False)
        .tail(1)
    )

    results = []
    for _, row in latest.iterrows():
        rec = recommend_price(
            model, row, category_dtypes,
            min_margin_pct=min_margin_pct,
            max_competitive_gap_pct=max_competitive_gap_pct)
        results.append({
            "product_id": rec.product_id,
            "store_id": rec.store_id,
            "current_price": rec.current_price,
            "current_quantity": rec.current_weekly_quantity,
            "current_margin": rec.current_weekly_margin,
            "recommended_price": round(rec.recommended_price),
            "predicted_quantity": round(rec.predicted_weekly_quantity),
            "predicted_margin": round(rec.predicted_weekly_margin) })

    out_df = pd.DataFrame(results)
    out_df.to_parquet(output_path, index=False)
    print(f"Wrote {len(out_df)} price recommendations to {output_path}")    

In [49]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("mode", choices=["train", "score"])
    parser.add_argument("--model-path", default="/tmp/demand_model.joblib")
    parser.add_argument("--output-path", default="/tmp/price_recommendations.parquet")
    parser.add_argument("--min-margin-pct", type=float, default=0.10)
    parser.add_argument("--max-competitive-gap-pct", type=float, default=0.05)
    args = parser.parse_args()

    if args.mode == "train":
        train_and_save(args.model_path)
    else:
        score_all(args.model_path, args.output_path, args.min_margin_pct, args.max_competitive_gap_pct)

usage: ipykernel_launcher.py [-h] [--model-path MODEL_PATH]
                             [--output-path OUTPUT_PATH]
                             [--min-margin-pct MIN_MARGIN_PCT]
                             [--max-competitive-gap-pct MAX_COMPETITIVE_GAP_PCT]
                             {train,score}
ipykernel_launcher.py: error: argument mode: invalid choice: '/home/docker/.local/share/jupyter/runtime/kernel-7f39a89d-19e2-438e-9a0a-4f75ea076784.json' (choose from 'train', 'score')


AssertionError: 

In [29]:
train_and_save(model_path= "demand_model_test.joblib")

/home/docker/.local/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001504 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2368
[LightGBM] [Info] Number of data points in the train set: 15000, number of used features: 22
[LightGBM] [Info] Start training from score 2.574422
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

In [22]:
score_all(
    model_path="demand_model_test.joblib",
    output_path="price_recommendations_test.parquet",
    min_margin_pct=0.10,
    max_competitive_gap_pct=0.05
)

/home/docker/.local/lib/python3.10/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


Wrote 200 price recommendations to price_recommendations_end.parquet


In [230]:
# تجميع البيانات بحساب المتوسط للأسعار والمجموع للكميات والأرباح
grouped_df = reco_df_e.groupby('product_id', as_index=False).agg({
    'current_price': 'mean',
    'current_quantity': 'sum' ,
    'current_margin': 'mean' ,
    'recommended_price': 'mean',
    'predicted_quantity': 'sum',
    'predicted_margin': 'sum'
})

# تقريب النتائج لأقرب رقم صحيح
#grouped_df = grouped_df.astype(int)
grouped_df

,product_id,current_price,current_quantity,current_margin,recommended_price,predicted_quantity,predicted_margin
0,100,95.3,26,0.226971,93.2,105,2107
1,101,101.7,35,0.231223,99.7,140,3120
2,102,103.9,36,0.228373,102.8,145,3268
3,103,110.8,19,0.219171,109.0,70,1470
4,104,113.2,46,0.216444,111.5,129,2999
5,105,119.6,25,0.232030,117.4,157,4102
6,106,123.8,9,0.234482,121.9,141,3799
7,107,126.1,49,0.212635,125.2,164,4198
8,108,135.7,29,0.235274,132.5,148,4317
9,109,136.2,16,0.231892,132.9,138,3835


In [207]:
# تحويل عمود current_margin من نسبة إلى أرباح بالجنيه
grouped_df['current_margin_amount'] = (
    grouped_df['current_quantity'] * grouped_df['current_price'] * grouped_df['current_margin']
)

# تقريب الناتج لرقم صحيح
grouped_df['current_margin_amount'] = grouped_df['current_margin_amount'].round(0).astype(int)
grouped_df

,product_id,current_price,current_quantity,current_margin,recommended_price,predicted_quantity,predicted_margin,current_margin_amount
0,100,95.3,26,0.226971,93.2,105,2107,562
1,101,101.7,35,0.231223,99.7,140,3120,823
2,102,103.9,36,0.228373,102.8,145,3268,854
3,103,110.8,19,0.219171,109.0,70,1470,461
4,104,113.2,46,0.216444,111.5,129,2999,1127
5,105,119.6,25,0.232030,117.4,157,4102,694
6,106,123.8,9,0.234482,121.9,141,3799,261
7,107,126.1,49,0.212635,125.2,164,4198,1314
8,108,135.7,29,0.235274,132.5,148,4317,926
9,109,136.2,16,0.231892,132.9,138,3835,505


In [23]:
reco_df_e= pd.read_parquet("price_recommendations_end.parquet")
reco_df_e

,product_id,store_id,current_price,current_quantity,current_margin,recommended_price,predicted_quantity,predicted_margin
0,119,10,169.0,2,0.201183,168,23,768
1,119,5,178.0,2,0.213483,177,30,1091
2,109,3,133.0,5,0.248120,128,25,695
3,119,2,179.0,0,0.206704,178,5,171
4,106,3,120.0,5,0.241667,124,1,34
...,...,...,...,...,...,...,...,...
195,112,10,175.0,2,0.268571,167,2,72
196,112,8,160.0,0,0.243750,157,4,154
197,112,2,158.0,2,0.221519,157,28,953
198,112,4,160.0,0,0.262500,157,3,135


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm

def _naive_elasticity_check_with_plot(df: pd.DataFrame):
    """
    يتخلص من الجزء الخاص بالرسم فقط لسهولة العرض.
    """
    # 1. إعداد النموذج (نفس الكود الأصلي)
    X = sm.add_constant(df["log_unit_retail"])
    y = df["log_quantity"]
    result = sm.OLS(y, X).fit()
    elasticity = float(result.params["log_unit_retail"])
    
    # 2. الرسم باستخدام Seaborn (سهل جداً)
    plt.figure(figsize=(10, 6))
    
    sns.regplot(
        data=df,
        x="log_unit_retail",
        y="log_quantity",
        ci=95,  # رسم منطقة الثقة 95%
        scatter_kws={"alpha": 0.3, "color": "teal"},  # شكل النقاط
        line_kws={"color": "red", "label": f"Elasticity (OLS): {elasticity:.2f}"}  # شكل الخط
    )
    
    plt.title("Price Elasticity Baseline: log(Price) vs log(Quantity)", fontsize=14)
    plt.xlabel("log(Price) [log_unit_retail]")
    plt.ylabel("log(Quantity) [log_quantity]")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.5)
    
    plt.show()

# مثال للاستخدام (افترض أن لديك DataFrame جاهز):
# _naive_elasticity_check_with_plot(your_dataframe)

# ------------------------------------------------------------------------------------------

# Deployment

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import warnings
from dataclasses import dataclass
import joblib
from lightgbm import LGBMRegressor ,early_stopping
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics import r2_score
from dremio_simple_query.connectv2 import DremioConnection
import argparse
import os
import seaborn as sns
import matplotlib.pyplot as plt
dremio = DremioConnection(
    location="grpc://dremio:32010",  
    username="mofah",
    password="fahmy12345",
)

In [ ]:
# utils.py


features = [
    # Categories 
    'category', 'sub_category', 'store_region',
    
    # Prices & Margins
    'log_unit_retail', 'avg_unit_cost', 'avg_competitor_price',
    'min_competitor_price', 'max_competitor_price',
    'competitor_price_ratio', 'price_difference', 'competitor_price_gap_pct',
    'current_profit_margin_pct', 'is_higher_than_competitor', 'is_undercut',
    
    # Lags & Rollings
    'quantity_lag_1_week', 'quantity_lag_4_weeks', 
    'quantity_rolling_avg_4_weeks', 'price_change_pct_weekly', 'price_rolling_avg_4_weeks',
    
    # Seasonality
    'month', 'quarter', 'week_of_year'
]

def connection():
    dremio = DremioConnection(
    location="grpc://dremio:32010",  
    username="mofah",
    password="fahmy12345")
    

def read_data(product_id: int) -> pd.DataFrame:
    
    query = f"""
        WITH ranked_sales AS (
            SELECT *,
                   ROW_NUMBER() OVER (
                    PARTITION BY store_id 
                    ORDER BY sales_week DESC ) as rn
            FROM nessie.marts.ml_dynamic_competitive_pricing
            WHERE product_id = {product_id} )
        SELECT *
        FROM ranked_sales
        WHERE rn = 1
        """
    df = dremio.toPandas(query)
    if "rn" in df.columns:
        df = df.drop(columns=["rn"])
    return df


def Feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values(['product_id', 'store_key', 'sales_week']).reset_index(drop=True)
    grouped = df.groupby(['product_id', 'store_key'])

    #df['log_quantity'] = np.log1p(df['quantity_sold'])
    df['log_unit_retail'] = np.log(df['avg_unit_retail'])
    #df['total_revenue'] = df['quantity_sold'] * df['avg_unit_retail']
    #df['units_per_transaction'] = df['quantity_sold'] / np.maximum(df['total_transactions'], 1)
    df['competitor_price_ratio'] = df['avg_unit_retail'] / df['avg_competitor_price']
    df['price_difference'] = df['avg_unit_retail'] - df['avg_competitor_price']
    df['competitor_price_gap_pct'] = df['price_difference'] / df['avg_competitor_price']

    # No fillna(0) - a missing margin is unknown, not zero. LightGBM
    # handles real NaN natively; manufacturing 0 just teaches the model
    # a false "no margin" signal for rows where margin is simply unknown.
    df['current_profit_margin_pct'] = (df['avg_unit_retail'] - df['avg_unit_cost']) / df['avg_unit_retail']
    #df['current_profit_margin_pct'] = df['quantity_sold'] * (df['avg_unit_retail'] - df['avg_unit_cost'])
    

    # np.nan, not pd.NA - keeps this a clean float64 column instead of
    # collapsing to object dtype, which is what caused the crash.
    df['is_higher_than_competitor'] = np.where(
        df['avg_competitor_price'].isna(), np.nan,
        (df['avg_unit_retail'] > df['avg_competitor_price']).astype(float)
    )
    df['is_undercut'] = np.where(
        df['min_competitor_price'].isna(), np.nan,
        (df['avg_unit_retail'] > df['min_competitor_price']).astype(float)
    )

    # No fillna(0) on lags - a real gap in history, left as NaN, is what
    # LightGBM is designed to split around correctly.
    df['quantity_lag_1_week'] = grouped['quantity_sold'].shift(1)
    df['quantity_lag_4_weeks'] = grouped['quantity_sold'].shift(4)
    df['price_change_pct_weekly'] = grouped['avg_unit_retail'].pct_change(periods=1)

    df['quantity_rolling_avg_4_weeks'] = (
        df.groupby(['product_id', 'store_key'])['quantity_sold']
          .transform(lambda s: s.shift(1).rolling(window=4, min_periods=1).mean())
    )
    # No fillna(df['avg_unit_retail']) - that fallback was silently
    # setting "4-week rolling avg price" equal to "today's price" for new
    # product-stores, artificially zeroing out any price-change signal
    # exactly where the model most needs to see "no history yet" honestly.
    df['price_rolling_avg_4_weeks'] = (
        df.groupby(['product_id', 'store_key'])['avg_unit_retail']
          .transform(lambda s: s.shift(1).rolling(window=4, min_periods=1).mean())
    )

    df['month'] = df['sales_week'].dt.month
    df['quarter'] = df['sales_week'].dt.quarter
    df['week_of_year'] = df['sales_week'].dt.isocalendar().week.astype(int)

    return df

def prepare_raw_data(df: pd.DataFrame) -> pd.DataFrame:
    float_cols = [
        'avg_unit_retail',
        'avg_unit_cost',
        'avg_competitor_price',
        'min_competitor_price',
        'max_competitor_price' ]
    for col in float_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)

    
    df['sales_week'] = pd.to_datetime(df['sales_week'])

    return df




def load_model(model_path: str) -> tuple:
    """Returns (model, category_dtypes). Use this instead of joblib.load directly."""
    bundle = joblib.load(model_path)
    return bundle["model"], bundle["category_dtypes"]



def fix_prediction_dtypes(rows: pd.DataFrame, category_dtypes: dict) -> pd.DataFrame:
    """
    base_row.to_frame().T turns a single pandas Series into a one-row
    DataFrame - and since a Series holds one dtype for its whole length,
    EVERY column comes out as generic `object`, not just the categoricals.
    LightGBM rejects object-dtyped numeric columns outright (as our own
    test caught). Two separate fixes needed:
    1. Categorical columns get the exact dtype (same categories, same
       order) captured at training time - re-inferring categories fresh
       from a single repeated row would only ever see one category level,
       which LightGBM either rejects or, worse, could silently map to the
       wrong code if it doesn't reject it.
    2. Every other feature column gets coerced back to numeric explicitly -
       object dtype survives even for what were originally float/int/bool
       values, since the Series-transpose step erases that distinction.
    """
    rows = rows.copy()
    for col, dtype in category_dtypes.items():
        rows[col] = rows[col].astype(dtype)

    numeric_columns = [c for c in features if c not in category_dtypes]
    for col in numeric_columns:
        rows[col] = pd.to_numeric(rows[col], errors="raise")

    return rows  


def recommend_price(model,base_row: pd.Series,category_dtypes: dict,min_margin_pct: float = 0.10,max_competitive_gap_pct: float = 0.05,n_candidates: int = 50) -> PriceRecommendation:
    """
    base_row: one row of engineered features for a specific
    (product_key, store_key), at its most recent known state - i.e. the
    latest row from pricing_features for that pair. All non-price features
    are held fixed at their current values; only price is varied across
    the candidate grid.

    Constraints, both enforced as hard filters (candidates violating either
    are excluded outright, not merely penalized):
    - min_margin_pct: price must clear unit_cost by at least this margin.
      This is a floor, not a target - it exists to prevent the optimizer
      from ever recommending a price that loses money or breaks a
      contractual minimum margin, regardless of what the demand curve says
      would maximize predicted quantity.
    - max_competitive_gap_pct: price must stay within this fraction of the
      last known competitor price. This is a business guardrail, not
      something derived from the demand model - it exists because pure
      margin-maximization without a competitive constraint can recommend
      prices far above market that the demand model has never actually
      observed and is extrapolating into blindly.
    """
    unit_cost = base_row["avg_unit_cost"]
    competitor_price = base_row["avg_competitor_price"]
    current_price = base_row["avg_unit_retail"]

    price_floor = unit_cost * (1 + min_margin_pct)
    if pd.notna(competitor_price):
        competitive_low = competitor_price * (1 - max_competitive_gap_pct)
        competitive_high = competitor_price * (1 + max_competitive_gap_pct)
    else:
        # No competitor price known for this product - fall back to a
        # wider band around current price rather than silently ignoring
        # the constraint entirely. Flag this explicitly in the output
        # rather than pretending the constraint was meaningfully applied.
        competitive_low = current_price * 0.85
        competitive_high = current_price * 1.15

    grid_low = max(price_floor, competitive_low)
    grid_high = max(grid_low * 1.01, competitive_high)  # guard against inverted/degenerate range

    candidate_prices = np.linspace(grid_low, grid_high, n_candidates)

    rows = pd.concat([base_row.to_frame().T] * n_candidates, ignore_index=True)
    rows["log_unit_retail"] = np.log(candidate_prices)
    rows["avg_unit_retail"] = candidate_prices
    rows['competitor_price_ratio'] = candidate_prices / rows['avg_competitor_price']
    rows['price_difference'] = candidate_prices - rows['avg_competitor_price']
    rows['competitor_price_gap_pct'] = rows['price_difference'] / rows['avg_competitor_price']
    rows['current_profit_margin_pct'] = (candidate_prices - rows['avg_unit_cost']) / candidate_prices
    
   
    rows['is_higher_than_competitor'] = np.where(
        rows['avg_competitor_price'].isna(), np.nan,
        ( candidate_prices > rows['avg_competitor_price']).astype(float))
    rows['is_undercut'] = np.where(
        rows['min_competitor_price'].isna(), np.nan,
        (candidate_prices > rows['min_competitor_price']).astype(float))    
    
    rows = fix_prediction_dtypes(rows, category_dtypes)

    predicted_log_quantity = model.predict(rows[features])
    predicted_quantity = np.expm1(predicted_log_quantity).clip(min=0)
    predicted_margin = predicted_quantity * (candidate_prices - unit_cost)
    

    candidates_df = pd.DataFrame({
        "price": candidate_prices,
        "predicted_quantity": predicted_quantity,
        "predicted_margin": predicted_margin})

    best_idx = candidates_df["predicted_margin"].idxmax()
    best = candidates_df.loc[best_idx]
 
    return PriceRecommendation(
        product_id=base_row["product_id"],
        store_id=base_row["store_id"],
        current_price=current_price,
        current_weekly_quantity=base_row["quantity_sold"],
        current_weekly_margin= base_row["current_profit_margin_pct"],
        recommended_price=float(best["price"]),
        predicted_weekly_quantity=float(best["predicted_quantity"]),
        predicted_weekly_margin=float(best["predicted_margin"]),
        candidates=candidates_df)



def score_all(model_path: str,df: pd.DataFrame, output_path: str, min_margin_pct: float, max_competitive_gap_pct: float) -> None:
    if not os.path.exists(model_path):
        raise FileNotFoundError(
            f"No trained model found at {model_path}. Run train_and_save "
            "first - scoring should never silently fall back to training "
            "inline, since that would retrain on a schedule meant only for "
            "scoring, hiding when the model actually last changed."
        )
    model, category_dtypes = load_model(model_path)
 
    df_pre = prepare_raw_data(df)
    features_df = Feature_engineering(df_pre)
    

    # Score using each product-store's most recent row only - recommending
    # a price is inherently a "given today's context" decision, not
    # something to backfill across history.
    latest = (
        features_df.sort_values("sales_week")
        .groupby(["product_id", "store_id"], as_index=False)
        .tail(1)
    )

    results = []
    for _, row in latest.iterrows():
        rec = recommend_price(
            model, row, category_dtypes,
            min_margin_pct=min_margin_pct,
            max_competitive_gap_pct=max_competitive_gap_pct)
        results.append({
            "product_id": rec.product_id,
            "store_id": rec.store_id,
            "current_price": rec.current_price,
            "current_quantity": rec.current_weekly_quantity,
            "current_margin": rec.current_weekly_margin,
            "recommended_price": round(rec.recommended_price),
            "predicted_quantity": round(rec.predicted_weekly_quantity),
            "predicted_margin": round(rec.predicted_weekly_margin) })

    out_df = pd.DataFrame(results)
    out_df.to_parquet(output_path, index=False)
    print(f"Wrote {len(out_df)} price recommendations to {output_path}") 

In [ ]:
# main 
import joblib
import os, time
import numpy as np
from fastapi import FastAPI, Form, HTTPException

In [ ]:
## Load the Model
model_path = os.path.join(os.getcwd(), 'models', 'forest_model.pkl')


app = FastAPI()

@app.post('/churn_prediction')
async def predict_demand(product_id : int,min_margin_pct: float, max_competitive_gap_pct: float):
    df= read_data(product_id)
    score_all(df=df, min_margin_pct =min_margin_pct , max_competitive_gap_pct = max_competitive_gap_pct )
    



In [33]:
os.getcwd()

'/home/docker/notebooks'

In [34]:
os.path.join(os.getcwd())

'/home/docker/notebooks'

In [1]:
pip list

Package              Version
-------------------- ------------
anyio                3.6.2
argon2-cffi          21.3.0
argon2-cffi-bindings 21.2.0
asttokens            2.2.0
attrs                22.1.0
backcall             0.2.0
beautifulsoup4       4.11.1
bleach               5.0.1
certifi              2026.7.22
cffi                 1.15.1
charset-normalizer   3.4.9
contourpy            1.3.2
cycler               0.12.1
dbus-python          1.2.18
debugpy              1.6.4
decorator            5.1.1
defusedxml           0.7.1
dremio_simple_query  2.0.0
duckdb               1.5.5
entrypoints          0.4
executing            1.2.0
fastjsonschema       2.16.2
fonttools            4.63.0
idna                 3.4
ipykernel            6.17.1
ipython              8.7.0
ipython-genutils     0.2.0
jedi                 0.18.2
Jinja2               3.1.2
joblib               1.5.3
jsonschema           4.17.3
jupyter_client       7.4.7
jupyter_core         5.1.0
jupyter-server       1.23.3
jupyte